In [ ]:
import jax
import os

In [ ]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

In [ ]:
from datetime import datetime
import functools
import itertools
import logging
import numpy as np
import pickle as pkl
import qiskit as qk
from qiskit import pulse
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
import qutip as qt
import re 
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm
from tqdm import tqdm
import warnings

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()
dt = backend.configuration().dt * ns
desired_1qb_duration = 225  # dt
desired_2qb_duration = 225 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

# Calibrate

In [ ]:
# Load shapes
fname_1qb = "../pico-pulses/saved-pulses-2024-02-16/a_single_qubit_gateset_R1e-5.csv"
pico_a = ps.from_csv(fname_1qb)
pico_a = np.vstack([pico_a, np.ones(pico_a.shape[1])])

names_1qb = ["X1", "X2", "SX1", "SX2", "I1", "I2"]
angles_1qb = [1.5 * np.pi, 0.5 * np.pi, 1.25 * np.pi, 0.25 * np.pi, 1.0, 0.0]

In [ ]:
cal_1qb_robust = ps.calibrate_gateset(
    gateset=pico_a,
    names=names_1qb,
    angles=angles_1qb,
    duration=duration_1qb,
    dt=dt,
    samples=32,
)

In [ ]:
cal_1qb_default = ps.default_single_qubit_gateset(
    names_1qb,
    duration_1qb,
    dt,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

ax = axes[0]
for i, (k, v) in enumerate(cal_1qb_robust.items()):
    ax.plot(ps.get_samples(v).real, label=names_1qb[i])
ax.legend(loc="upper right")

ax = axes[1]
for i, (k, v) in enumerate(cal_1qb_default.items()):
    ax.plot(ps.get_samples(v).real, label=names_1qb[i])
ax.legend(loc="upper right")

In [ ]:
# Load shapes
def fname(k):
    return f"../pico-pulses/saved-pulses-2024-01-16/a_two_qubit_{k}.csv"

base_2qb = {k: ps.from_csv(fname(k))[0] for k in ["default", "robust"]}
robust_2qb = 2 * [base_2qb["robust"]]
default_2qb = 2 * [base_2qb["default"]]
names_2qb = ["+", "-"]

In [ ]:
cal_2qb_robust = ps.calibrate_gateset(
    gateset=robust_2qb,
    names=names_2qb,
    angles=[0.125 * np.pi, -0.125 * np.pi],
    duration=duration_2qb,
    dt=dt,
    samples=32,
)

In [ ]:
cal_2qb_default = ps.calibrate_gateset(
    gateset=default_2qb,
    names=names_2qb,
    angles=[0.125 * np.pi, -0.125 * np.pi],
    duration=duration_2qb,
    dt=dt,
    samples=32,
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
for i, (k, v) in enumerate(cal_2qb_robust.items()):
    ax.plot(ps.get_samples(v).real, label=names_2qb[i])
ax.legend(loc="upper right")

# Build circuit

In [ ]:
registers = [0, 1]
coupling_map = ps.backend_edges(backend)

In [ ]:
circ = qk.QuantumCircuit(2, 2)

In [ ]:
def prepare_state(circ, registers):
    [circ.sx(r) for r in registers]

def apply_xy4(circ, q):
    circ.x(q)
    circ.y(q)
    circ.x(q)
    circ.y(q)

def apply_simultaneous_xy4(circ, q0, q1):
    apply_xy4(circ, q0)
    apply_xy4(circ, q1)

In [ ]:
circ = qk.QuantumCircuit(2, 2)

prepare_state(circ, registers)
for _ in range(1):
    apply_simultaneous_xy4(circ, 0, 1)
prepare_state(circ, registers)

# circ.draw('mpl')


In [ ]:
mb = ps.MomentBuilder(coupling_map=ps.backend_edges(backend))

moments = mb.build_moments(
    circ, 
    optimization_level=0, 
)

In [ ]:
moments

In [ ]:
sched = ps.build_pulse_schedule(
    moments,
    qubit_gates=cal_1qb_default,
    cr_gates=cal_2qb_default,
    registers=registers,
    backend=backend,
    name="default-hamiltonian-simulation",
    dt=dt,
    insert_identities=True,
    verbose=True
)

robust_sched = ps.build_pulse_schedule(
    moments,
    qubit_gates=cal_1qb_robust,
    cr_gates=cal_2qb_robust,
    registers=registers,
    backend=backend,
    name="robust-hamiltonian-simulation",
    dt=dt,
    insert_identities=True,
    verbose=False
)

# Check durations
assert robust_sched.duration == sched.duration

# Would be nice to disable phase channels when plotting
# sched.draw()